# Operator sensitivity — separate diagnostic experiment

> **Revision 2026-09-23.** Two earlier runs (20260921T060241Z_796df2a1 and
> 20260921T175730Z_51e704d2) stopped at seed-0 feature selection with zero
> fits. The runner compared a shifted-feature list in candidate order with the
> archive's alphabetically sorted copy. The repaired runner checks both orders
> correctly, keeps every tolerance unchanged, and saves the selection evidence
> before comparing. This notebook refuses any package without that runner.

The completed feature audit found directional packet maxima above directional
byte totals in **58,242 of 74,127 transformed flows**, plus **17,354 zero-byte-rate
cases** under the principal footprint. The counts overlap. This experiment asks
whether the C2 degradation and reselected C3 result depend on those rules.

**This notebook trains models: 60–150 fits, with a maximum of 150.** It reuses a
C3 fit across arms only when the ordered feature list and training rows are
identical. Allow the run to finish. Original results and source code are preserved.
This is an additional diagnostic, not an approved replacement for the paper's
main experiment or a claim that all physical-consistency issues are solved.

1. Select **Runtime → Run all** in Colab.
2. Upload the **new PQC_IDS_Reproduction_Working_Copy.zip supplied with this notebook**.
3. Connect the Drive containing the original four CSV files.
4. Return **PQC_IDS_Operator_Sensitivity_Results.zip**, including any failure logs.

The CSV folder defaults to `/content/drive/MyDrive/cicids2018`; change `DATA_DIR`
only if necessary. A new result ZIP is also copied to
`MyDrive/pqc_ids_operator_sensitivity`. The export includes probability arrays
and will be larger than the preceding audit ZIP. The single code cell runs
setup, analysis and export together. Do not re-run it merely to export results;
if the browser download fails, the completed ZIP is in the printed Drive folder.

## Four fixed conditions

| Arm | Difference from the historical operator |
|---|---|
| historical | None |
| byte_lengths | Set added packet lengths to the allocated bytes per added packet; update directional/overall minima and maxima |
| zero_byte_rate | Recompute byte rate for originally zero-byte eligible records using the original packet rate and unchanged duration relationship |
| combined | Apply both changes |

The main hybrid768 footprint still adds 1,184 forward bytes, 1,088 backward bytes
and one packet per direction. Therefore the declared added packet lengths are
1,184 and 1,088 bytes. This is a synthetic allocation assumption, not a measured
packet trace. Byte totals, packet/header additions, eligibility and timing stay
as in the historical experiment.

Each split uses the original 70:30 stratification and model settings. KS/Holm
and MI selection are recomputed using training data for each arm. C1 and the
four C2 arms share the same fitted baseline classifier. MLP scaling uses only
the relevant training columns. All RF predictions use one worker, fixed in
advance because the preceding diagnostic identified order-sensitive ties.
This is a new diagnostic setting; it does not replace historical predictions.

## Checks before fitting and interpretation

The dataset hashes and historical shifted matrix must match the verified run.
The combined arm must pass the specified checks for newly introduced length
bounds and nonpositive rates before any classifier is fitted. Every split also
checks the historical arm's feature selection against the locked main record.
Failure stops the run and exports the evidence.

The outputs retain split-level scores, paired changes, selected features,
model-sharing IDs and all prediction/probability arrays. Summaries are descriptive;
no new equivalence decision or independent-split confidence interval is asserted.
The unresolved overlapping-split uncertainty issue still applies.

The alternatives repair only specified feature relationships. They do not
reconstruct packet traces, TLS framing, timing, standard deviations or subflows.
The current scope is the main representation with hybrid768; it makes no new
claim about the B3 representation or other footprints. The full fixed plan is
`environment/operator_sensitivity_plan.json` in the supplied package.

This notebook and diagnostic code were prepared with assistant assistance.

In [1]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import stat
import subprocess
import sys
import tempfile
import traceback
import uuid
import zipfile
from google.colab import files, drive

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
SESSION = Path(tempfile.mkdtemp(prefix="pqc_operator_sensitivity_", dir="/content"))

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Edit this folder only if the four original CSV files moved.
DATA_DIR = Path("/content/drive/MyDrive/cicids2018")
DRIVE_OUTPUT = SESSION / "records"
DRIVE_OUTPUT.mkdir()
RUN_OUTPUT = DRIVE_OUTPUT / "results"
SUMMARY = {"run_id": RUN_ID, "status": "preparing", "fits_completed": 0,
           "scope": "Separate operator sensitivity, preserving historical records",
           "historical_results_modified": False, "colab_kernel_python": platform.python_version()}
DRIVE_CONNECTED = False

def save_summary():
    (DRIVE_OUTPUT / "notebook_status.json").write_text(json.dumps(SUMMARY, indent=2) + "\n")

save_summary()
try:
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one updated working-copy ZIP.")
    archive_path = SESSION / "working_copy.zip"
    archive_path.write_bytes(uploaded[zip_names[0]])
    del uploaded
    extracted = SESSION / "package"
    with zipfile.ZipFile(archive_path) as archive:
        if archive.testzip() is not None:
            raise ValueError("The ZIP failed its integrity check. Download the working copy again.")
        for info in archive.infolist():
            path = PurePosixPath(info.filename)
            if path.is_absolute() or ".." in path.parts or stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError("Unexpected archive path: " + info.filename)
        archive.extractall(extracted)
    manifest_paths = list(extracted.glob("*/package_manifest.json"))
    if len(manifest_paths) != 1:
        raise ValueError("Expected one working-copy package manifest.")
    PACKAGE = manifest_paths[0].parent
    manifest = json.loads(manifest_paths[0].read_text())
    listed = set()
    for item in manifest["files"]:
        path = (PACKAGE / item["path"]).resolve()
        if not path.is_relative_to(PACKAGE.resolve()):
            raise ValueError("Invalid manifest path.")
        if not path.is_file() or path.stat().st_size != item["bytes"] or sha256(path) != item["sha256"]:
            raise ValueError("Package file does not match the manifest: " + item["path"])
        listed.add(item["path"])
    actual = {p.relative_to(PACKAGE).as_posix() for p in PACKAGE.rglob("*") if p.is_file()}
    if actual != listed | {"package_manifest.json"} or len(listed) != len(manifest["files"]):
        raise ValueError("The archive contains unlisted files or duplicate manifest records.")
    if not (PACKAGE / "scripts/run_operator_sensitivity.py").is_file():
        raise ValueError("This is an older working copy. Use the ZIP provided with this notebook.")
    # The repaired runner (revision 2026-09-23-archive-order-fix) is required.
    # Older packages compared a candidate-ordered shifted list with the
    # alphabetically sorted archive and stopped before any fit.
    EXPECTED_RUNNER_SHA256 = "f3865d2e4897f624e58836bc58aa9283ba50e58d32ef68a2a21ddbd29aa04760"
    RUNNER_SHA256 = sha256(PACKAGE / "scripts/run_operator_sensitivity.py")
    if RUNNER_SHA256 != EXPECTED_RUNNER_SHA256:
        raise ValueError("This working copy has an older sensitivity runner. Use the ZIP supplied with this notebook.")
    print("Repaired sensitivity runner verified:", RUNNER_SHA256)
    print("Working-copy files verified:", len(listed))
    print("Run identifier:", RUN_ID)

    SUMMARY["package_sha256"] = sha256(archive_path)
    SUMMARY["runner_sha256"] = RUNNER_SHA256
    SUMMARY["package_manifest_sha256"] = sha256(PACKAGE / "package_manifest.json")
    shutil.copy2(PACKAGE / "package_manifest.json", DRIVE_OUTPUT / "working_copy_manifest.json")
    for name in ("main_known_versions.json", "main_requirements.txt", "operator_sensitivity_plan.json"):
        shutil.copy2(PACKAGE / "environment" / name, DRIVE_OUTPUT / name)
    TARGET = json.loads((PACKAGE / "environment/main_known_versions.json").read_text())
    drive.mount("/content/drive")
    DRIVE_CONNECTED = True
    SUMMARY["data_directory"] = str(DATA_DIR)
    SUMMARY["target"] = TARGET
    save_summary()

    RUN_ENV = os.environ.copy()
    for name in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV"):
        RUN_ENV.pop(name, None)
    RUN_ENV["PYTHONNOUSERSITE"] = "1"
    RUN_ENV["UV_PYTHON_INSTALL_DIR"] = str(SESSION / "python_runtimes")
    RUN_ENV["UV_CACHE_DIR"] = str(SESSION / "uv_cache")

    def run_logged(command, log_name, env):
        with (DRIVE_OUTPUT / log_name).open("w", encoding="utf-8") as log:
            log.write("Command: " + json.dumps([str(s) for s in command]) + "\n")
            log.flush()
            with subprocess.Popen([str(s) for s in command], stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                  text=True, bufsize=1, env=env) as process:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                return process.wait()

    ENVIRONMENT_READY = False
    VENV = SESSION / "operator_sensitivity_environment"
    FIT_PYTHON = VENV / "bin/python"
    try:
        tooling = SESSION / "uv_tooling"
        bootstrap = [sys.executable, "-m", "pip", "install", "--target", tooling,
                     "--no-deps", "uv==0.12.15"]
        if run_logged(bootstrap, "01_uv_install.log", RUN_ENV) != 0:
            raise RuntimeError("Could not install the pinned uv tool.")
        uv_env = RUN_ENV.copy()
        uv_env["PYTHONPATH"] = str(tooling)
        uv = [sys.executable, "-m", "uv"]
        create = uv + ["venv", "--python", TARGET["python"], "--managed-python", VENV]
        if run_logged(create, "02_python_setup.log", uv_env) != 0:
            raise RuntimeError("Could not create the requested Python runtime.")
        install = uv + ["pip", "install", "--python", FIT_PYTHON, "--only-binary", ":all:",
                        "-r", PACKAGE / "environment/main_requirements.txt"]
        if run_logged(install, "03_packages_install.log", uv_env) != 0:
            raise RuntimeError("Could not install the declared package versions.")
        ENVIRONMENT_READY = True
        SUMMARY["status"] = "environment_installed"
    except Exception as error:
        SUMMARY["status"] = "environment_setup_failed"
        SUMMARY["error"] = str(error)
        (DRIVE_OUTPUT / "setup_traceback.txt").write_text(traceback.format_exc())
        print("Setup failed:", error)
    finally:
        save_summary()
    print("Environment ready:", ENVIRONMENT_READY)

    if not ENVIRONMENT_READY:
        raise RuntimeError("Sensitivity did not run because environment setup failed; return the exported logs.")
    SUMMARY["status"] = "running_sensitivity"
    save_summary()
    returncode = run_logged([FIT_PYTHON, "-u", PACKAGE / "scripts/run_operator_sensitivity.py",
                             "--data-dir", DATA_DIR, "--output", RUN_OUTPUT,
                             "--cache-dir", SESSION / "input_cache"],
                            "04_operator_sensitivity.log", RUN_ENV)
    SUMMARY["runner_returncode"] = returncode
    status_path = RUN_OUTPUT / "sensitivity_status.json"
    if not status_path.is_file():
        raise RuntimeError("Sensitivity runner produced no status record; return the ZIP.")
    audit_status = json.loads(status_path.read_text())
    SUMMARY["sensitivity_status"] = audit_status["status"]
    SUMMARY["fits_completed"] = audit_status.get("fits_completed", 0)
    if returncode != 0 or audit_status["status"] != "sensitivity_completed" or audit_status.get("metric_rows") != 270 or audit_status.get("completed_splits") != list(range(10)):
        raise RuntimeError("Sensitivity did not complete; return the ZIP containing its failure evidence.")
    SUMMARY["status"] = "sensitivity_completed"
    print("Completed fits:", audit_status["fits_completed"], "(maximum 150)")
    print("Metric and prediction records:", audit_status["metric_rows"])
    print("Return the ZIP for independent interpretation; no historical scores were replaced.")

except BaseException as error:
    SUMMARY["status"] = "sensitivity_incomplete"
    SUMMARY["error"] = str(error)
    (DRIVE_OUTPUT / "notebook_traceback.txt").write_text(traceback.format_exc())
    print("Please return the ZIP with this recorded issue:", error)
finally:
    SUMMARY["exported_utc"] = datetime.now(timezone.utc).isoformat()
    save_summary()
    result_files = sorted(p for p in DRIVE_OUTPUT.rglob("*") if p.is_file())
    items = [{"path": p.relative_to(DRIVE_OUTPUT).as_posix(), "bytes": p.stat().st_size,
              "sha256": sha256(p)} for p in result_files]
    manifest_path = DRIVE_OUTPUT / "result_manifest.json"
    manifest_path.write_text(json.dumps({"run_id": RUN_ID, "files": items}, indent=2) + "\n")
    result_files.append(manifest_path)
    result_zip = SESSION / "PQC_IDS_Operator_Sensitivity_Results.zip"
    with zipfile.ZipFile(result_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in result_files:
            archive.write(path, path.relative_to(DRIVE_OUTPUT).as_posix())
    with zipfile.ZipFile(result_zip) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("Result archive failed its integrity check.")
    runner_status = RUN_OUTPUT / "sensitivity_status.json"
    if runner_status.is_file():
        captured = json.loads(runner_status.read_text())
        print("Runner recorded fits:", captured.get("fits_completed", 0))
    if DRIVE_CONNECTED:
        try:
            saved = Path("/content/drive/MyDrive/pqc_ids_operator_sensitivity") / RUN_ID
            saved.mkdir(parents=True, exist_ok=False)
            shutil.copy2(result_zip, saved / result_zip.name)
            print("Drive copy:", saved / result_zip.name)
        except OSError as error:
            print("Drive copy failed, but the downloadable ZIP is available:", error)
    print("Final status:", SUMMARY["status"])
    print("Fits recorded:", SUMMARY.get("fits_completed", "see runner status"))
    print("ZIP SHA-256:", sha256(result_zip))
    files.download(str(result_zip))


Saving PQC_IDS_Reproduction_Working_Copy.zip to PQC_IDS_Reproduction_Working_Copy.zip
Repaired sensitivity runner verified: f3865d2e4897f624e58836bc58aa9283ba50e58d32ef68a2a21ddbd29aa04760
Working-copy files verified: 270
Run identifier: 20260922T180508Z_1f6bbb0b
Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 62.6 MB/s eta 0:00:00
Using CPython 3.12.13
Creating virtual environment at: pqc_operator_sensitivity_k9972jrd/operator_sensitivity_environment
Activate with: source pqc_operator_sensitivity_k9972jrd/operator_sensitivity_environment/bin/activate
Using Python 3.12.13 environment at: pqc_operator_sensitivity_k9972jrd/operator_sensitivity_environment
Resolved 21 packages in 383ms
Prepared 21 packages in 12.98s
Installed 21 packages in 155ms
 + cloudpickle==3.1.2
 + formulaic==1.2.2
 + interface-meta==2.0.1
 + joblib==1.6.0
 + narwhals==2.26.0
 + numpy==2.0.2
 + nvidia-nccl-cu12==2.31.2
 + packaging==26.3
 + pandas==2.2.2
 + patsy==1.0.3
 + python-d

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# Split the finished results ZIP into 20 MB parts for upload. Does not rerun anything.
from pathlib import Path
import hashlib, json
from google.colab import drive, files

drive.mount("/content/drive")
src = Path("/content/drive/MyDrive/pqc_ids_operator_sensitivity/20260922T180508Z_1f6bbb0b/PQC_IDS_Operator_Sensitivity_Results.zip")
out = src.parent / "parts"
out.mkdir(exist_ok=True)

data = src.read_bytes()
whole = hashlib.sha256(data).hexdigest()
print("Full ZIP:", len(data), "bytes, SHA-256", whole)
assert whole == "9832e073d015236e54cd789d99b072ee3b7b749b11599681bc43a179dcbdb041", "Not the ZIP from run 20260922T180508Z_1f6bbb0b"

size = 20 * 1024 * 1024
record = {"source": src.name, "bytes": len(data), "sha256": whole, "parts": []}
for i in range(0, len(data), size):
    part = out / f"results.zip.part{i // size + 1:02d}"
    part.write_bytes(data[i:i + size])
    record["parts"].append({"name": part.name, "bytes": part.stat().st_size,
                            "sha256": hashlib.sha256(part.read_bytes()).hexdigest()})
(out / "parts_manifest.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))

for p in sorted(out.iterdir()):
    files.download(str(p))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full ZIP: 125015009 bytes, SHA-256 9832e073d015236e54cd789d99b072ee3b7b749b11599681bc43a179dcbdb041
{
  "source": "PQC_IDS_Operator_Sensitivity_Results.zip",
  "bytes": 125015009,
  "sha256": "9832e073d015236e54cd789d99b072ee3b7b749b11599681bc43a179dcbdb041",
  "parts": [
    {
      "name": "results.zip.part01",
      "bytes": 20971520,
      "sha256": "5e08e815e6d885f2824346600a48092dd1b8737a7fc7bc9dc9faddce3cf0cd39"
    },
    {
      "name": "results.zip.part02",
      "bytes": 20971520,
      "sha256": "3bfa25065d9315907874f91c40562cb8c9af184a3f1b398d769fdd45368e7dfc"
    },
    {
      "name": "results.zip.part03",
      "bytes": 20971520,
      "sha256": "eccf41ac71bd60a7e4ac3584335365b1c40f268bcce1cd1afe6af2078c5a7a96"
    },
    {
      "name": "results.zip.part04",
      "bytes": 20971520,
      "sha256": "057b7d7fa625334b74df3ca92943884caaba6a2d427

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (5 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# Split the full experiment notebook into 20 MB parts for upload. Does not run the notebook.
from pathlib import Path
import hashlib, json
from google.colab import drive, files

# If the notebook is in Google Drive, put its path here.
# If it is on your computer, set this to None and you will be asked to upload it.
NOTEBOOK = None

if NOTEBOOK is not None:
    drive.mount("/content/drive")
    data = NOTEBOOK.read_bytes()
    name = NOTEBOOK.name
else:
    up = files.upload()
    name, data = next(iter(up.items()))

out = Path("/content/notebook_parts"); out.mkdir(exist_ok=True)
whole = hashlib.sha256(data).hexdigest()
print(name, len(data), "bytes, SHA-256", whole)

size = 20 * 1024 * 1024
record = {"source": name, "bytes": len(data), "sha256": whole, "parts": []}
for i in range(0, len(data), size):
    part = out / f"notebook.part{i // size + 1:02d}"
    part.write_bytes(data[i:i + size])
    record["parts"].append({"name": part.name, "bytes": part.stat().st_size,
                            "sha256": hashlib.sha256(part.read_bytes()).hexdigest()})
(out / "notebook_parts_manifest.json").write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))
for p in sorted(out.iterdir()):
    files.download(str(p))

Saving full_experiment.ipynb to full_experiment.ipynb
full_experiment.ipynb 99333363 bytes, SHA-256 da2f9b54e5a58d8955aae7a33d443a8d194904343ca9a2525e5b493fbc4eb5bd
{
  "source": "full_experiment.ipynb",
  "bytes": 99333363,
  "sha256": "da2f9b54e5a58d8955aae7a33d443a8d194904343ca9a2525e5b493fbc4eb5bd",
  "parts": [
    {
      "name": "notebook.part01",
      "bytes": 20971520,
      "sha256": "0d0d94887a90104f1fb924736e1eac001f2115282ac9aa67f0e47231dee55527"
    },
    {
      "name": "notebook.part02",
      "bytes": 20971520,
      "sha256": "a37a80150dd51ab5eb51dc6138cd46b3406cd7d3458fbd313e8ae1ffa8d50b21"
    },
    {
      "name": "notebook.part03",
      "bytes": 20971520,
      "sha256": "94ec5351119c8f12c0fdc64140ffc03ff71fe3d415e0adcd57987c99770c5c14"
    },
    {
      "name": "notebook.part04",
      "bytes": 20971520,
      "sha256": "9915006855b6ca6427cffc1934f44fb29c409a239a8ddd4718d5f5733800774d"
    },
    {
      "name": "notebook.part05",
      "bytes": 15447283,
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (3 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# Collect the small source files needed to verify Table 4.10 and Table 4.11.
# Reads only; changes nothing in Drive. Downloads one ZIP (split into 20 MB parts if larger).
from pathlib import Path
import hashlib, json, zipfile
from google.colab import drive, files

drive.mount("/content/drive")
MY = Path("/content/drive/MyDrive")

FOLDERS = {  # whole folders, files up to 50 MB each
    "combined_timing_identifier": MY / "final_thesis_experiment_package/combined_robustness_analysis/timing_free_identifier_free",
    "timing_free_v6": MY / "pqc_ids_dissertation/final_experiment_package_v6/01_timing_free",
}
V5_ZIP = MY / "final_thesis_package/final_experiment_package_2026_07_02_v5_merged.zip"
V5_MEMBERS = [
    "final_experiment_package_2026_07_02_v5_merged/01_raw_results/revised_pipeline_b3_mlp_sweep_raw.json",
    "final_experiment_package_2026_07_02_v5_merged/01_raw_results/revised_pipeline_b3_mlp_sweep_summary.json",
]
LIMIT = 50 * 1024 * 1024

def sha(data): return hashlib.sha256(data).hexdigest()
out = Path("/content/table410_sources"); out.mkdir(exist_ok=True)
bundle = out / "table410_sources.zip"
record = {"folders": {}, "v5_zip": {}, "skipped": [], "missing": []}
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for name, folder in FOLDERS.items():
        if not folder.is_dir():
            record["missing"].append(str(folder)); continue
        items = []
        for p in sorted(folder.rglob("*")):
            if not p.is_file(): continue
            if p.stat().st_size > LIMIT:
                record["skipped"].append({"path": str(p), "bytes": p.stat().st_size}); continue
            data = p.read_bytes()
            arc = f"{name}/{p.relative_to(folder).as_posix()}"
            z.writestr(arc, data)
            items.append({"path": arc, "bytes": len(data), "sha256": sha(data)})
        record["folders"][name] = {"source": str(folder), "files": items}
    if V5_ZIP.is_file():
        raw = V5_ZIP.read_bytes()
        record["v5_zip"] = {"source": str(V5_ZIP), "bytes": len(raw), "sha256": sha(raw), "members": []}
        with zipfile.ZipFile(V5_ZIP) as v5:
            for m in V5_MEMBERS:
                try: data = v5.read(m)
                except KeyError: record["missing"].append(m); continue
                arc = "v5_members/" + m.split("/")[-1]
                z.writestr(arc, data)
                record["v5_zip"]["members"].append({"member": m, "path": arc, "bytes": len(data), "sha256": sha(data)})
        del raw
    else:
        record["missing"].append(str(V5_ZIP))
    z.writestr("collection_record.json", json.dumps(record, indent=2))

data = bundle.read_bytes()
print("Files collected:", sum(len(f["files"]) for f in record["folders"].values()) + len(record["v5_zip"].get("members", [])))
print("Missing:", record["missing"] or "none")
print("Skipped (>50 MB):", record["skipped"] or "none")
print("Bundle:", len(data), "bytes, SHA-256", sha(data))
size = 20 * 1024 * 1024
parts = []
if len(data) <= size:
    parts = [bundle]
else:
    for i in range(0, len(data), size):
        p = out / f"table410_sources.zip.part{i // size + 1:02d}"
        p.write_bytes(data[i:i + size]); parts.append(p)
    (out / "parts_manifest.json").write_text(json.dumps({"bytes": len(data), "sha256": sha(data),
        "parts": [{"name": p.name, "bytes": p.stat().st_size, "sha256": sha(p.read_bytes())} for p in parts]}, indent=2))
    parts.append(out / "parts_manifest.json")
for p in parts:
    files.download(str(p))

Mounted at /content/drive
Files collected: 80
Missing: none
Skipped (>50 MB): none
Bundle: 210143 bytes, SHA-256 548fb64620db91ccc98d5bfa35d979d8611ba3bb84ae738de86ac18b36a281a1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# Collect source files for the pipeline-component and random-stable controls.
# Reads only; changes nothing in Drive. Downloads one ZIP (20 MB parts if larger).
from pathlib import Path
import hashlib, json, zipfile
from google.colab import drive, files

drive.mount("/content/drive")
MY = Path("/content/drive/MyDrive")
V6 = MY / "pqc_ids_dissertation/final_experiment_package_v6"
PKG = MY / "final_thesis_experiment_package"

FOLDERS = {  # whole folders; files over 50 MB are listed but not copied
    "component_controls_v6": V6 / "04_pipeline_component_controls",
    "random_stable_d6": PKG / "combined_robustness_analysis/repeated_random_stable_subsets",
}
GLOBS = {  # single files matched by pattern
    "v6_tables": (V6 / "tables", "v6_component_control_*"),
    "thesis_tables": (PKG / "thesis_tables", "d6_*"),
    "records": (PKG / "experiment_records", "d6_*"),
}
LIMIT = 50 * 1024 * 1024

def sha(data): return hashlib.sha256(data).hexdigest()
out = Path("/content/component_sources"); out.mkdir(exist_ok=True)
bundle = out / "component_sources.zip"
record = {"groups": {}, "skipped": [], "missing": []}

def add(z, group, path, arc):
    if path.stat().st_size > LIMIT:
        record["skipped"].append({"path": str(path), "bytes": path.stat().st_size}); return
    data = path.read_bytes()
    z.writestr(arc, data)
    record["groups"].setdefault(group, []).append({"path": arc, "source": str(path), "bytes": len(data), "sha256": sha(data)})

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for name, folder in FOLDERS.items():
        if not folder.is_dir():
            record["missing"].append(str(folder)); continue
        for p in sorted(folder.rglob("*")):
            if p.is_file() and "__pycache__" not in p.parts:
                add(z, name, p, f"{name}/{p.relative_to(folder).as_posix()}")
    for name, (folder, pattern) in GLOBS.items():
        hits = sorted(folder.glob(pattern)) if folder.is_dir() else []
        if not hits:
            record["missing"].append(f"{folder}/{pattern}"); continue
        for p in hits:
            if p.is_file():
                add(z, name, p, f"{name}/{p.name}")
    z.writestr("collection_record.json", json.dumps(record, indent=2))

data = bundle.read_bytes()
print("Files collected:", sum(len(v) for v in record["groups"].values()))
print("Missing:", record["missing"] or "none")
print("Skipped (>50 MB):", record["skipped"] or "none")
print("Bundle:", len(data), "bytes, SHA-256", sha(data))
size = 20 * 1024 * 1024
parts = [bundle]
if len(data) > size:
    parts = []
    for i in range(0, len(data), size):
        p = out / f"component_sources.zip.part{i // size + 1:02d}"
        p.write_bytes(data[i:i + size]); parts.append(p)
    (out / "parts_manifest.json").write_text(json.dumps({"bytes": len(data), "sha256": sha(data),
        "parts": [{"name": p.name, "bytes": p.stat().st_size, "sha256": sha(p.read_bytes())} for p in parts]}, indent=2))
    parts.append(out / "parts_manifest.json")
for p in parts:
    files.download(str(p))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files collected: 1096
Missing: none
Skipped (>50 MB): none
Bundle: 1637767 bytes, SHA-256 614c9b3d18debd293e5c660411c6b589a76d58047d21e7abf252cfc30cd6df9e


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# Collect source files for the duplicate audits and source-day analyses.
# Reads only; changes nothing in Drive. Downloads one ZIP (20 MB parts if larger).
from pathlib import Path
import hashlib, json, zipfile
from google.colab import drive, files

drive.mount("/content/drive")
MY = Path("/content/drive/MyDrive")
PKG = MY / "final_thesis_experiment_package"

FOLDERS = {  # whole folders; files over 50 MB are listed but not copied
    "duplicate_audit": PKG / "combined_robustness_analysis/duplicate_audit",
    "source_day_robustness": PKG / "combined_robustness_analysis/source_day_robustness",
}
GLOBS = {  # single files matched by pattern
    "thesis_tables_d5b": (PKG / "thesis_tables", "d5b_*"),
    "thesis_tables_dup": (PKG / "thesis_tables", "*duplicate*"),
    "records_d4": (PKG / "experiment_records", "d4*"),
    "records_d5": (PKG / "experiment_records", "d5*"),
    "records_dup": (PKG / "experiment_records", "*duplicate*"),
}
LIMIT = 50 * 1024 * 1024

def sha(data): return hashlib.sha256(data).hexdigest()
out = Path("/content/dup_sourceday_sources"); out.mkdir(exist_ok=True)
bundle = out / "dup_sourceday_sources.zip"
record = {"groups": {}, "skipped": [], "missing": []}

def add(z, group, path, arc):
    if path.stat().st_size > LIMIT:
        record["skipped"].append({"path": str(path), "bytes": path.stat().st_size}); return
    data = path.read_bytes()
    z.writestr(arc, data)
    record["groups"].setdefault(group, []).append({"path": arc, "source": str(path), "bytes": len(data), "sha256": sha(data)})

with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for name, folder in FOLDERS.items():
        if not folder.is_dir():
            record["missing"].append(str(folder)); continue
        for p in sorted(folder.rglob("*")):
            if p.is_file() and "__pycache__" not in p.parts:
                add(z, name, p, f"{name}/{p.relative_to(folder).as_posix()}")
    for name, (folder, pattern) in GLOBS.items():
        hits = sorted(folder.glob(pattern)) if folder.is_dir() else []
        if not hits:
            record["missing"].append(f"{folder}/{pattern}"); continue
        for p in hits:
            if p.is_file():
                add(z, name, p, f"{name}/{p.name}")
    z.writestr("collection_record.json", json.dumps(record, indent=2))

data = bundle.read_bytes()
print("Files collected:", sum(len(v) for v in record["groups"].values()))
print("Missing:", record["missing"] or "none")
print("Skipped (>50 MB):", record["skipped"] or "none")
print("Bundle:", len(data), "bytes, SHA-256", sha(data))
size = 20 * 1024 * 1024
parts = [bundle]
if len(data) > size:
    parts = []
    for i in range(0, len(data), size):
        p = out / f"dup_sourceday_sources.zip.part{i // size + 1:02d}"
        p.write_bytes(data[i:i + size]); parts.append(p)
    (out / "parts_manifest.json").write_text(json.dumps({"bytes": len(data), "sha256": sha(data),
        "parts": [{"name": p.name, "bytes": p.stat().st_size, "sha256": sha(p.read_bytes())} for p in parts]}, indent=2))
    parts.append(out / "parts_manifest.json")
for p in parts:
    files.download(str(p))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Files collected: 189
Missing: none
Skipped (>50 MB): none
Bundle: 122416694 bytes, SHA-256 2ca4ff52b587f99736cf15eeaf7e7a92961f8946e3ca6f93d1a21bb0723396ab


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (5 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>